# Review marked sites

Gathers **every landslide-candidate GeoJSON you've saved** (from any notebook, any location) onto one map for review, with the same BEFORE / TOPO views to re-inspect each spot. Re-run cell 1 anytime to pick up newly saved files.

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

DATA = (Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent) / "data"
files = sorted((DATA / "landslide_candidates").glob("*.geojson"))

frames = []
for f in files:
    gdf = gpd.read_file(f)
    gdf["source"] = f.stem
    frames.append(gdf)
sites = pd.concat(frames, ignore_index=True) if frames else None

if sites is None:
    print("No saved candidates yet — draw and export in notebooks 01/02/03 first.")
else:
    summary = sites.groupby("source").agg(
        features=("geometry", "size"),
        types=("geometry", lambda s: dict(s.geom_type.value_counts())),
    )
    print(f"{len(sites)} features from {len(files)} file(s)")
summary

44 features from 1 file(s)


,features,types
source,,
la-guaira_20260626_150535_17_3010,44,"{'Point': 26, 'Polygon': 18}"


## All sites on one map — click to edit

- **Yellow** = your marked candidates. **Click one to select it** (turns red), then use the
  **Site editor panel (top-right)** to change its tag/note (*Apply tag*) or remove it (*Delete site*)
- Switch **BEFORE / AFTER / TOPO** with the pinned buttons — AFTER shows the post-event
  scene(s) your candidates were marked on
- Edits live in memory until you run the save cell below

In [2]:
import leafmap

from geer_venezuela import (
    ATTRIBUTION,
    add_site_editor,
    HILLSHADE,
    HILLSHADE_ATTRIBUTION,
    WAYBACK_ATTRIBUTION,
    WAYBACK_PRE_EVENT,
    add_compare_control,
    asset_href,
    load_items,
)

items = load_items("post-event")

m = leafmap.Map()
m.add_tile_layer(HILLSHADE, name="TOPO — terrain hillshade", attribution=HILLSHADE_ATTRIBUTION)
m.add_tile_layer(
    WAYBACK_PRE_EVENT,
    name="BEFORE — Esri Wayback 2026-05-28",
    attribution=WAYBACK_ATTRIBUTION,
    max_zoom=19,
)

# add the AFTER scene(s) the candidates were marked on — the scene id is in each filename
after_layers = []
marked_ids = {item_id for item_id in items["id"] if any(item_id in f.stem for f in files)}
for item_id in sorted(marked_ids):
    scene = items[items["id"] == item_id].iloc[0]
    layer_name = f"AFTER — {scene['title']}"
    m.add_cog_layer(
        asset_href(scene, "visual"),
        name=layer_name,
        attribution=ATTRIBUTION,
        zoom_to_layer=False,
    )
    after_layers.append(layer_name)

footprints = items[["id", "title", "location", "geometry"]]
m.add_gdf(
    footprints,
    layer_name="Post-event scene footprints",
    style={"color": "#ff3b30", "weight": 1.5, "fillOpacity": 0},
    zoom_to_layer=False,
)
add_site_editor(m, sites)
m.fit_bounds([[sites.total_bounds[1], sites.total_bounds[0]], [sites.total_bounds[3], sites.total_bounds[2]]])

views = {"BEFORE": "BEFORE — Esri Wayback 2026-05-28"}
if after_layers:
    views["AFTER"] = after_layers
views["TOPO"] = "TOPO — terrain hillshade"
add_compare_control(m, views, selected="AFTER" if after_layers else "BEFORE")
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

## Save changes — GeoJSON + KMZ

Writes your edits back to the per-source GeoJSON files **(overwriting them)** and produces
**`landslide_candidates.kmz`** — GEER's standard KML-style product; open it in Google Earth
or attach it to the reconnaissance report. Placemarks are named `site NNN — category` with
source and notes in the description.

In [3]:
from geer_venezuela import save_sites

written = save_sites(
    m.sites_gdf,
    geojson_dir=DATA / "landslide_candidates",
    kmz_file=DATA / "landslide_candidates" / "landslide_candidates.kmz",
)
for w in written:
    print("wrote", w)

wrote /Users/lornearnold/GitHub/GEER_Venezuela/data/landslide_candidates/la-guaira_20260626_150535_17_3010.geojson
wrote /Users/lornearnold/GitHub/GEER_Venezuela/data/landslide_candidates/landslide_candidates.kmz


## Site list with coordinates and BEFORE-image dates

One row per marked feature — note this table reduces **polygons to their centroid lat/lon** (the polygons themselves stay intact in the GeoJSON files). Includes tags (once features are tagged via notebook 01) and the actual capture date of the pre-event imagery under each site. The capture-date lookup makes one web request per site, so this takes a few seconds.

In [4]:
from geer_venezuela import wayback_capture_date

table = m.sites_gdf.copy()  # reflects your edits above
points = table.geometry.centroid
table["lat"] = points.y.round(5)
table["lon"] = points.x.round(5)
table["type"] = table.geometry.geom_type

before = [wayback_capture_date(lat, lon) for lat, lon in zip(table["lat"], table["lon"])]
table["before_date"] = [b["captured"] for b in before]
table["before_res_m"] = [b["resolution_m"] for b in before]

columns = ["source", "type", "lat", "lon", "before_date", "before_res_m"]
for extra in ("note", "category"):  # present once sites are tagged (notebook 01)
    if extra in table.columns:
        columns.insert(1, extra)

site_list = table[columns]
site_list.to_csv(DATA / "landslide_candidates" / "site_list.csv", index_label="site_no")
print(f"Saved {DATA / 'landslide_candidates' / 'site_list.csv'}")
site_list

Saved /Users/lornearnold/GitHub/GEER_Venezuela/data/landslide_candidates/site_list.csv


,source,category,note,type,lat,lon,before_date,before_res_m
0,la-guaira_20260626_150535_17_3010,None,None,Point,10.54902,-66.98999,2025-02-20,0.34
1,la-guaira_20260626_150535_17_3010,None,None,Point,10.54354,-66.98793,2025-03-02,0.50
2,la-guaira_20260626_150535_17_3010,None,None,Point,10.53676,-66.98143,2025-03-02,0.50
3,la-guaira_20260626_150535_17_3010,None,None,Point,10.53385,-66.98363,2025-03-02,0.50
4,la-guaira_20260626_150535_17_3010,None,None,Point,10.52622,-66.99678,2025-03-02,0.50
5,la-guaira_20260626_150535_17_3010,None,None,Point,10.52639,-66.99422,2025-03-02,0.50
6,la-guaira_20260626_150535_17_3010,None,None,Point,10.52461,-66.99927,2025-03-02,0.50
7,la-guaira_20260626_150535_17_3010,None,None,Point,10.52112,-67.00239,2025-03-02,0.50
8,la-guaira_20260626_150535_17_3010,None,None,Polygon,10.52052,-66.99532,2025-03-02,0.50
9,la-guaira_20260626_150535_17_3010,None,None,Polygon,10.49090,-67.01972,2025-03-02,0.50


## Other saved products

Everything else you've exported lives next to the candidates and drags straight into QGIS/ArcGIS:

```
data/landslide_candidates/   your marked sites (+ site_list.csv from above)
data/routes/                 watch segments, arterials, corridor summary
data/terrain/                steep-area polygons, geology
data/usgs/                   USGS rapid assessment grid
```